# A CAM physics scheme as a function

`mmacro_pcond` is CAM5's cloud macrophysics.  In the model it runs inside a
timestep on a chunk of columns, fed from the physics buffer.  Here it is
just a function: one vertical column in, that column's outputs back, the
original Fortran doing the arithmetic.  No `fc.Driver`, no MPI, no model
state -- the routine is linked into its own small image and runs in a worker
process beside this notebook.

Five parts: load the function, read its signature, call it by hand, define
a sampling space over inputs and parameters, generate and save training data.


## 1. Load the function


In [ ]:
import json
import numpy as np
import freecam as fc

f = fc.physics.load_function("mmacro_pcond")
f


## 2. The signature

Inputs are one column: `[lev]` profiles and scalars.  In/out arguments take
an initial value and come back updated.  Parameters are the scheme's own
tunables; each is an extra dimension a surrogate can learn.


In [ ]:
print(f.describe())


## 3. Call it by hand

A real column, taken from a captured model call (`validation/` keeps it
with its provenance), makes a sensible starting point.  Change any input or
parameter and call again; every call is independent.


In [ ]:
anchor = json.load(open(fc.physics.function.REPO / "validation/pi_cam_mmacro_pcond_anchor_column.json"))
column = {name: np.asarray(value) for name, value in anchor["inputs"].items()}

result = f.run(column)
print(result.status)
print("cloud fraction by level:", np.round(result["cld"], 3))
print("net condensation rate, max:", result["qme"].max(), "kg/kg/s")


In [ ]:
# A warmer column with a lower low-cloud RH threshold.
warmer = {**column, "t0": column["t0"] + 2.0}
tuned = f.run(warmer, parameters={"cldfrc_rhminl": 0.80})
print(tuned.status, "| cloud fraction changed at levels:", np.nonzero(tuned["cld"] != result["cld"])[0])


In [ ]:
# Inputs the scheme cannot resolve end in a Fortran abort; the worker
# restarts and the function keeps working.
bad = f.run({**column, "t0": np.full(30, 1.0)})
print(bad.status, "|", bad.message)
print(f.run(column).status)


## 4. A sampling space

Each drawn input or parameter gets a distribution.  Pressure is never drawn
level by level: `HybridPressure` draws one surface pressure and builds the
whole profile through the hybrid coordinate, so `p` and `dp` stay
consistent.  The thermodynamic profiles are anchored on the real column
with bounded noise, and one parameter joins the space as its own dimension.


In [ ]:
hybrid = anchor["hybrid_coordinate"]
surface = float(column["p"][-1] + 0.5 * column["dp"][-1])

sampling_space = {
    "p": fc.physics.HybridPressure(
        np.asarray(hybrid["hyai"]), np.asarray(hybrid["hybi"]), hybrid["p0"],
        surface=fc.physics.Uniform(0.95 * surface, 1.05 * surface),
        produces=("p", "dp", "pint"),
    ),
    "t0": fc.physics.Anchored(column["t0"], scale=1.0),
    "qv0": fc.physics.Anchored(column["qv0"], scale=0.05, relative=True, clip=(0.0, None)),
    "ql0": fc.physics.Anchored(column["ql0"], scale=0.05, relative=True, clip=(0.0, None)),
    "qi0": fc.physics.Anchored(column["qi0"], scale=0.05, relative=True, clip=(0.0, None)),
    "cldfrc_rhminl": fc.physics.Uniform(0.80, 0.95),
}
fixed = {name: value for name, value in column.items() if name not in sampling_space and name not in ("dp",)}
print(fc.physics.SamplingSpace(f.spec, sampling_space).describe())


## 5. Generate and save training data

One sample is one column and one call.  A sample the scheme refuses keeps
its inputs and a status; it is never written as data.  The file records the
image hash, the module-state digest and the seed, so any row can be
re-executed.


In [ ]:
dataset = f.generate_dataset(500, sampling_space, seed=42, inputs=fixed)
print(len(dataset), "samples:", dataset.status_counts)
print("inputs:", len(dataset.inputs), "| parameters:", list(dataset.parameters), "| outputs:", len(dataset.outputs))


In [ ]:
path = dataset.save("mmacro_pcond_training.nc")
reloaded = fc.physics.Dataset.load(path)
sample = reloaded.sample(0)
again = f.run(sample["inputs"], sample["parameters"])
print("row 0 re-executes to its stored output:", np.array_equal(again["cld"], reloaded.outputs["cld"][0]))


In [ ]:
f.close()
